In [1]:
!pip install openai fastapi uvicorn langchain pydantic

Imports & Client

In [2]:
from openai import OpenAI
import json
client = OpenAI()

Intent & Entity Extraction

In [3]:
def extract_intent_entities(user_input: str):
    prompt = f"""
    Extract intent,utterance and entities from: "{user_input}"
    Return JSON with keys: intent, entities.
    """
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)
extract_intent_entities("Book compliance training next week")

{'intent': 'BookTraining',
 'utterance': 'Book compliance training next week',
 'entities': {'training_type': 'compliance', 'date': 'next week'}}

Context Manager

In [4]:
class ContextManager:
    def __init__(self):
        self.state = {}

    def update_context(self, user_id, intent, entities,utterance):
        if user_id not in self.state:
            self.state[user_id] = []
        self.state[user_id].append({"intent": intent, "entities": entities,"utterance":utterance})
        return self.state[user_id]

context = ContextManager()

Router Layer

In [5]:
def route_intent(intent, entities,utterance):
    if intent == "book_training":
        return {"status": "success", "details": f"Training booked: {entities}"}
    elif intent == "update_leave":
        return {"status": "success", "details": f"Leave updated: {entities}"}
    else:
        return {"status": "unknown_intent"}

 Response Generator

In [6]:
def generate_response(intent, entities, backend_result,utterance):
    if backend_result["status"] == "success":
        return f"✅ Your request for {intent} with {entities} has been processed."
    else:
        return f"⚠️ I couldn’t process your request. Please refine your input."

Full Pipeline Function

In [11]:
def clu_pipeline(user_id: str, message: str):
    parsed = extract_intent_entities(message)
    print(parsed)
    context_state = context.update_context(user_id, parsed["intent"], parsed["entities"],parsed["utterance"])
    backend_result = route_intent(parsed["intent"], parsed["entities"],parsed["utterance"])
    print(backend_result)
    response = generate_response(parsed["intent"], parsed["entities"], backend_result,parsed["utterance"])
    return {"response": response, "context": context_state}

# Example Run
clu_pipeline("user123", "I want to book compliance training next week")
clu_pipeline("user123", "The room was excellent, but the service was terrible.")

{'intent': 'book_training', 'utterance': 'I want to book compliance training next week', 'entities': {'training_type': 'compliance', 'time': 'next week'}}
{'status': 'success', 'details': "Training booked: {'training_type': 'compliance', 'time': 'next week'}"}
{'intent': 'ProvideFeedback', 'utterance': 'The room was excellent, but the service was terrible.', 'entities': {'room': 'excellent', 'service': 'terrible'}}
{'status': 'unknown_intent'}


{'response': '⚠️ I couldn’t process your request. Please refine your input.',
 'context': [{'intent': 'book_training',
   'entities': {'training_type': 'compliance training', 'time': 'next week'},
   'utterance': 'I want to book compliance training next week'},
  {'intent': 'BookTraining',
   'entities': {'training_type': 'compliance training', 'time': 'next week'},
   'utterance': 'I want to book compliance training next week'},
  {'intent': 'provide_feedback',
   'entities': {'room_condition': 'excellent', 'service': 'terrible'},
   'utterance': 'The room was excellent, but the service was terrible.'},
  {'intent': 'book_training',
   'entities': {'training_type': 'compliance', 'time': 'next week'},
   'utterance': 'I want to book compliance training next week'},
  {'intent': 'ProvideFeedback',
   'entities': {'room': 'excellent', 'service': 'terrible'},
   'utterance': 'The room was excellent, but the service was terrible.'}]}